# Custom dataset

A dataset assembled from parts, and what asks it.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install cloud-volume deltalake 'fsspec[http]' navis numpy pandas pyarrow

import re
import copy
import pandas as pd
import numpy as np
import navis
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather
import pyarrow.compute as pc
import fsspec
from cloudvolume import CloudVolume
from deltalake import DeltaTable, QueryBuilder

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_ids(df, *columns):
    """Cast id columns to text, so 18-digit neuron ids stay exact."""
    if df is None:
        return df
    for name in columns:
        if name not in df.columns:
            continue
        col = df[name]
        df[name] = (
            col.astype('Int64').astype('string')
            if pd.api.types.is_numeric_dtype(col)
            else col.astype('string')
        )
    return df


def coda_id_column(values):
    """Convert an id column to exact text, with <NA> where a value is not a valid id.

    Mirrors `idText` in Coda's core/ids.ts. Blanks, fractional floats and floats above
    2**53 become <NA>: a float that large has already been rounded to a different id.
    """
    column = pd.Series(values)
    codes, uniques = pd.factorize(column.array)
    if len(uniques) * 2 < len(column):
        # Cast each distinct value once; an edge list repeats a neuron once per partner.
        text = coda_id_column(pd.Series(uniques)).array
        return pd.Series(pd.api.extensions.take(text, codes, allow_fill=True), index=column.index)
    if pd.api.types.is_float_dtype(column):
        column = column.mask((column % 1 != 0) | (column.abs() > 2**53))
    elif pd.api.types.infer_dtype(column, skipna=True) in ('floating', 'mixed-integer-float', 'mixed'):
        # Same rule for floats mixed with other values (e.g. 10003.0 in a CSV column with
        # blanks), without turning 18-digit integers into floats.
        column = column.map(lambda v: v if not isinstance(v, float) else
                            int(v) if v.is_integer() and abs(v) <= 2**53 else None)
    text = coda_ids(pd.DataFrame({'id': column}), 'id')['id']
    if pd.api.types.is_numeric_dtype(column):
        return text
    text = text.str.strip()
    return text.mask(text == '')


def coda_typed_ids(ids, integer):
    """Return ids for matching against a column: as text, or as integers for an integer column."""
    text = coda_id_column(pd.Series(list(ids), dtype=object)).dropna()
    if integer:
        return [int(i) for i in text[text.str.fullmatch(r'-?\d+')]]
    return list(text)


def coda_is_id_name(name):
    """Check whether a column name marks an id column, i.e. its last word is `id` or `ids`."""
    spaced = re.sub(r'([a-z0-9])([A-Z])', r'\1 \2', name)
    words = [w for w in re.split(r'[^A-Za-z0-9]+', spaced) if w]
    return bool(words) and words[-1].lower() in ('id', 'ids')


class CodaTableFile:
    """Read a Parquet or Feather file, or a Delta table, the way Coda's Link Table does.

    `read` fetches only the requested columns and, given `where` ({column: ids}), only the
    rows matching those ids in all columns (or in any, with any_of=True). Parquet and Delta
    files are skipped using their statistics, so a lookup in a sorted synapse table reads
    only a fraction of it. A Delta table is read at `version`.

    A lookup with an empty id list returns an empty frame. Id columns come back as exact
    text so that 18-digit ids survive; pass text=False to keep them as stored.
    """

    def __init__(self, location, format='parquet', text_columns=None, version=None):
        self.location = location
        self.format = format
        # The Delta table version the canvas read, so later commits do not change the result.
        self.version = version
        self.text_columns = None if text_columns is None else set(text_columns)
        self._schema = None
        self._remote = None
        self._table = None
        # Filter Table conditions as (columns, keep), where keep maps a frame to a row mask.
        self.conditions = []

    def where(self, columns, keep):
        """Return a copy with a row condition added, as a Filter Table after a Link Table.

        Nothing is read here; later reads drop the rows that fail the condition.
        """
        other = copy.copy(self)
        other.conditions = self.conditions + [(list(columns), keep)]
        return other

    def _open(self):
        # pyarrow reads gs:// and s3:// itself; https:// goes through fsspec, which serves
        # byte ranges. The handle is reused so the footer is fetched only once.
        if self.location.startswith(('http://', 'https://')):
            if self._remote is None:
                self._remote = fsspec.open(self.location, 'rb').open()
            return self._remote
        return self.location

    def schema(self):
        # Read once, without reading rows: from a Delta table's log or a file's footer
        # (Feather is Arrow's IPC file format).
        if self._schema is None:
            if self.format == 'delta':
                self._schema = pa.schema(self._delta().schema().to_arrow())
            elif self.format == 'parquet':
                self._schema = pq.read_schema(self._open())
            else:
                self._schema = pa.ipc.open_file(self._open()).schema
        return self._schema

    def _delta(self):
        # Public buckets are read anonymously, as on the canvas; delta-rs expects gs:// for GCS.
        if self._table is None:
            uri = self.location.replace('https://storage.googleapis.com/', 'gs://', 1)
            public = uri.startswith(('gs://', 's3://'))
            self._table = DeltaTable(uri, version=self.version,
                                     storage_options={'skip_signature': 'true'} if public else None)
        return self._table

    def _delta_batches(self, columns, clauses, any_of, limit=None):
        # Query with SQL; id lookups become IN clauses so delta-rs can skip files by their stats.
        quote = lambda name: '"' + name.replace('"', '""') + '"'
        literal = lambda v: str(v) if isinstance(v, int) else "'" + str(v).replace("'", "''") + "'"
        sql = 'SELECT ' + (', '.join(map(quote, columns)) if columns else '*') + ' FROM t'
        tests = [f'{quote(name)} IN ({", ".join(map(literal, values))})' for name, _, values in clauses if values]
        if tests:
            sql += ' WHERE ' + (' OR ' if any_of else ' AND ').join(tests)
        if limit:
            sql += f' LIMIT {int(limit)}'
        return pa.RecordBatchReader.from_stream(QueryBuilder().register('t', self._delta()).execute(sql))

    def names(self):
        return self.schema().names

    def read(self, columns=None, where=None, limit=None, text=True, any_of=False):
        # text=False keeps id columns as stored (grouping int64 is much faster than strings).
        # any_of=True matches rows with an id in any of the where columns instead of all.
        schema = self.schema()
        wanted = None if columns is None else list(dict.fromkeys(columns))
        tested = [c for cols, _ in self.conditions for c in cols]
        needed = None if wanted is None else list(dict.fromkeys(wanted + tested))
        clauses = [(name, 'in', coda_typed_ids(ids, pa.types.is_integer(schema.field(name).type)))
                   for name, ids in (where or {}).items()]
        if clauses and (all if any_of else any)(not values for _, _, values in clauses):
            table = schema.empty_table()
        elif not clauses and limit:
            # A limit without a lookup reads only the first blocks, as the canvas does.
            table = self._head(needed, limit)
        elif self.format == 'delta':
            table = self._kept(self._delta_batches(needed, clauses, any_of).read_all())
        elif self.format == 'parquet':
            filters = [[c] for c in clauses if c[2]] if any_of else clauses
            table = self._kept(pq.read_table(self._open(), columns=needed, filters=filters or None))
        else:
            names = None if needed is None else list(dict.fromkeys(needed + [c for c, _, _ in clauses]))
            table = feather.read_table(self._open(), columns=names, memory_map=True)
            masks = []
            for name, _, values in clauses:
                column = table[name]
                if pa.types.is_dictionary(column.type):
                    column = column.cast(column.type.value_type)
                # Cast (rather than construct) so text ids can match a float column.
                masks.append(pc.is_in(column, value_set=pa.array(values).cast(column.type)))
            if masks:
                mask = masks[0]
                for more in masks[1:]:
                    mask = (pc.or_ if any_of else pc.and_)(mask, more)
                table = table.filter(mask)
            table = self._kept(table)
        if wanted is not None:
            table = table.select(wanted)
        if limit:
            table = table.slice(0, limit)
        return (self._as_text(table) if text else table).to_pandas()

    def _as_text(self, table):
        # Cast with Arrow before pandas sees the column, so int64 ids convert exactly.
        for name in self._text_columns() & set(table.column_names):
            at = table.column_names.index(name)
            table = table.set_column(at, name, table[name].cast(pa.string()))
        return table

    def _head(self, columns, limit):
        batches, held = [], 0
        if self.format == 'delta':
            # Without conditions the limit goes into the query, so the scan stops early.
            source = self._delta_batches(columns, [], False, None if self.conditions else limit)
        elif self.format == 'parquet':
            source = pq.ParquetFile(self._open()).iter_batches(columns=columns)
        else:
            # Decode (and, over a URL, fetch) only the requested columns.
            names = self.schema().names
            fields = None if columns is None else [names.index(c) for c in columns]
            reader = pa.ipc.open_file(
                self._open(), options=pa.ipc.IpcReadOptions(included_fields=fields))
            source = (reader.get_batch(i) for i in range(reader.num_record_batches))
        for batch in source:
            # Count rows after applying conditions, as the canvas does.
            kept = self._kept(pa.Table.from_batches([batch]))
            batches.append(kept)
            held += kept.num_rows
            if held >= limit:
                break
        return pa.concat_tables(batches) if batches else self.schema().empty_table()

    def _kept(self, table):
        # Apply the conditions to ids as text (as the canvas does, regardless of `text`)
        # and drop the rows that fail any of them.
        if not self.conditions:
            return table
        names = list(dict.fromkeys(c for cols, _ in self.conditions for c in cols))
        frame = self._as_text(table.select(names)).to_pandas()
        mask = pd.Series(True, index=frame.index)
        for _, keep in self.conditions:
            mask &= keep(frame).fillna(False).astype(bool)
        return table.filter(pa.array(mask.to_numpy()))

    def _text_columns(self):
        # Use the columns the canvas listed; otherwise every integer column whose name marks
        # it as an id (Coda's own rule), so ids never arrive as int64.
        if self.text_columns is None:
            self.text_columns = {
                field.name for field in self.schema()
                if pa.types.is_integer(field.type) and coda_is_id_name(field.name)
            }
        return self.text_columns


class CodaFrameTable:
    """Wrap a pandas frame so it can be read like a `CodaTableFile`."""

    def __init__(self, frame):
        self.frame = frame
        self._keys = {}

    def names(self):
        return list(self.frame.columns)

    def _key(self, name):
        # Convert each key column to text once and cache it.
        if name not in self._keys:
            self._keys[name] = coda_id_column(self.frame[name])
        return self._keys[name]

    def read(self, columns=None, where=None, limit=None, text=True, any_of=False):
        # Select by position, not label: a frame may have repeated index labels.
        keep = None
        for name, ids in (where or {}).items():
            integer = pd.api.types.is_integer_dtype(self.frame[name])
            key = self.frame[name] if integer else self._key(name)
            hits = key.isin(coda_typed_ids(ids, integer)).to_numpy()
            keep = hits if keep is None else (keep | hits if any_of else keep & hits)
        frame = self.frame if keep is None else self.frame[keep]
        if columns is not None:
            frame = frame[list(dict.fromkeys(columns))]
        return frame.head(limit) if limit else frame


def coda_walk(seed_ids, direction, hops, one_hop):
    """Walk the connectome breadth-first from the seeds, as Coda's Connectivity node.

    `one_hop(ids, way)` returns the edges leaving `ids` in direction `way` ('downstream' or
    'upstream') with columns preId, preType, postId, postType and weight (ids as text).
    The result has the same columns plus `hop` and `direction`, each row oriented from
    presynaptic to postsynaptic.

    Notes:

    * Each neuron is expanded at most once, so recurrent loops do not repeat. Edges back
      into visited neurons are still reported.
    * An edge found again at a later hop keeps the hop and direction it was first found at.
    * direction="both" expands both ways at every hop, so it also finds neurons that share
      inputs with a seed.
    """
    columns = ['preId', 'preType', 'postId', 'postType', 'weight']
    ways = {'both': ['downstream', 'upstream'], 'inputs': ['upstream']}.get(direction, ['downstream'])
    frontier, expanded, seen = {str(i) for i in seed_ids}, set(), {}
    for hop in range(1, int(hops) + 1):
        todo = frontier - expanded
        if not todo:
            break
        expanded |= todo
        reached = set()
        for way in ways:
            for row in one_hop(todo, way).reindex(columns=columns).itertuples(index=False):
                key = (row.preId, row.postId)
                if key in seen:
                    # Found from both ends at the same hop: label it "both".
                    if seen[key][0] == hop and seen[key][1] != way:
                        seen[key][1] = 'both'
                    continue
                seen[key] = [hop, way, row]
                reached.add(row.postId if way == 'downstream' else row.preId)
        frontier = reached - expanded
    return pd.DataFrame([(*row, hop, way) for hop, way, row in seen.values()],
                        columns=columns + ['hop', 'direction'])


class CodaCustomDataset:
    """Coda's Custom Dataset: a dataset assembled from your own tables and files.

    `labels` is the neuron table (one row per neuron, keyed by text `neuronId`) or, if none
    was wired, every id in the edge list. `edges` and `synapses` each name a frame or a
    `CodaTableFile` plus the columns to read. Without an edge list, connectivity counts the
    synapse table's rows per pair. `meshes` and `skeletons` are functions of the ids.
    """

    def __init__(self, neurons=None, id_column='neuronId', edges=None, synapses=None,
                 meshes=None, skeletons=None):
        self._labels = None if neurons is None else self.neuron_table(neurons, id_column)
        self._labels_derived = neurons is None
        self.edges = self._part(edges)
        self.synapses_spec = self._part(synapses)
        self.mesh_source = meshes
        self.skeleton_source = skeletons
        self._edge_list = None
        self._type_map = None

    @staticmethod
    def _part(spec):
        if spec is None or isinstance(spec['source'], CodaTableFile):
            return spec
        return {**spec, 'source': CodaFrameTable(spec['source'])}

    @staticmethod
    def neuron_table(frame, id_column):
        """Turn a table into the neuron table, keyed by `id_column`.

        The id column becomes the first column, `neuronId`, as text. Rows without a valid id
        are dropped; for repeated ids the first row is kept.
        """
        ids = coda_id_column(frame[id_column]).values
        rest = [c for c in frame.columns if c not in (id_column, 'neuronId')]
        table = frame[rest].copy()
        table.insert(0, 'neuronId', ids)
        return table[table['neuronId'].notna()].drop_duplicates(subset='neuronId').reset_index(drop=True)

    # ---- neurons ---------------------------------------------------------------------------

    @property
    def labels(self):
        if self._labels is None:
            if self.edges is None and self.synapses_spec is None:
                raise ValueError(
                    'This Custom Dataset has no neuron table, so it cannot say which neurons it '
                    'has. Wire a table into Neurons, or an edge list into Edges.'
                )
            edges = self.edge_list()
            ids = pd.unique(pd.concat([edges['preId'], edges['postId']], ignore_index=True))
            self._labels = pd.DataFrame({'neuronId': ids})
        return self._labels

    def lookup(self, ids):
        """Return the neuron table rows for these ids.

        Without a neuron table, the ids are returned as given.
        """
        wanted = coda_typed_ids(ids, False)
        if self._labels_derived:
            return pd.DataFrame({'neuronId': list(dict.fromkeys(wanted))})
        return self.labels[self.labels['neuronId'].isin(wanted)].reset_index(drop=True)

    def _types(self):
        if self._type_map is None:
            # Without a neuron table there are no types (and none are looked up).
            typed = not self._labels_derived and 'type' in self.labels.columns
            self._type_map = (dict(zip(self.labels['neuronId'], self.labels['type']))
                              if typed else {})
        return self._type_map

    # ---- connectivity ----------------------------------------------------------------------

    def edge_list(self):
        """Return every connection as preId, postId, weight, with ids as text.

        Repeated pairs are summed. Without a weight column (a synapse table, or an edge list
        that has no such column) each row counts as one.
        """
        if self._edge_list is None:
            spec = self.edges or self.synapses_spec
            if spec is None:
                raise ValueError('This Custom Dataset has no connectivity. Wire an edge list '
                                 'into its Edges socket, or a synapse table into Synapses.')
            source = spec['source']
            weight = self.edges.get('weight') if self.edges else None
            if weight is not None and weight not in source.names():
                print(f'The edge list has no column {weight!r}, so each row counts as one.')
                weight = None
            keys = [spec['pre'], spec['post']]
            frame = source.read(columns=keys + ([weight] if weight else []), text=False)
            # Aggregate raw pairs first so ids are converted once per edge, not once per row.
            if weight:
                frame = frame.assign(**{weight: pd.to_numeric(frame[weight], errors='coerce')})
                pairs = frame.groupby(keys, sort=False)[weight].sum(min_count=1)
            else:
                pairs = frame.groupby(keys, sort=False).size()
            pairs = pairs.reset_index(name='weight')
            edges = pd.DataFrame({
                'preId': coda_id_column(pairs[keys[0]]),
                'postId': coda_id_column(pairs[keys[1]]),
                'weight': pairs['weight'],
            }).dropna()
            if all(pd.api.types.is_integer_dtype(pairs[k]) for k in keys):
                self._edge_list = edges.reset_index(drop=True)
            else:
                # Merge pairs whose ids differed only in spelling (e.g. 1 and ' 1').
                self._edge_list = edges.groupby(['preId', 'postId'], as_index=False, sort=False)['weight'].sum()
        return self._edge_list

    def connectivity(self, seed_ids, direction, hops=1, min_weight=1, include_fragments=False):
        """Find partners over the edge list, as Coda's Connectivity node does.

        Without include_fragments, partners must be in the neuron table (seeds are exempt).
        Without a neuron table, every partner is kept.
        """
        edges = self.edge_list()
        edges = edges[edges['weight'] >= min_weight]
        seeds = set(coda_typed_ids(seed_ids, False))
        keep = (None if include_fragments or self._labels_derived
                else set(self.labels['neuronId']) | seeds)
        types = self._types()

        def one_hop(ids, way):
            near, far = ('preId', 'postId') if way == 'downstream' else ('postId', 'preId')
            step = edges[edges[near].isin(ids)]
            if keep is not None:
                step = step[step[far].isin(keep)]
            return step.assign(preType=step['preId'].map(types), postType=step['postId'].map(types))

        return coda_walk(seeds, direction, hops, one_hop)

    # ---- synapses --------------------------------------------------------------------------

    def _synapse_rows(self, where, carry=True, any_of=False):
        """Return synapse rows matching {'pre': ids, 'post': ids}: both ends, or either
        with any_of=True."""
        spec = self.synapses_spec
        if spec is None:
            raise ValueError('This Custom Dataset has no synapses: wire a synapse table into its '
                             'Synapses socket.')
        carried = list(spec['carry']) if carry else []
        # Skip carried columns whose names clash with the ones written here (e.g. a raw
        # voxel `z` would overwrite the one in nanometres).
        clashing = [c for c in carried if c in ('pre', 'post', 'x', 'y', 'z')]
        if clashing:
            print(f'Not carried, their names being taken by the position and ends: {clashing}')
            carried = [c for c in carried if c not in clashing]
        frame = spec['source'].read(
            columns=[spec['pre'], spec['post'], *spec['position'], *carried],
            where={spec[side]: ids for side, ids in where.items()},
            any_of=any_of,
        )
        (vx, vy, vz), (x, y, z) = spec['voxel'], spec['position']
        return pd.DataFrame({
            'pre': coda_id_column(frame[spec['pre']]).values,
            'post': coda_id_column(frame[spec['post']]).values,
            'x': frame[x].astype(float).values * vx,
            'y': frame[y].astype(float).values * vy,
            'z': frame[z].astype(float).values * vz,
            **{c: frame[c].values for c in carried},
        })

    def synapses(self, ids, polarity=None):
        """Fetch the synapses of a set of neurons.

        neuronId is the queried neuron, partnerId the other end, x/y/z are in nm, followed by
        any carried columns. Both polarities are returned unless `polarity` is given.
        """
        sides = [polarity] if polarity else ['pre', 'post']
        # Read both ends at once, then split by end. Ids are matched as text and as numbers,
        # since an integer column was matched numerically.
        rows = self._synapse_rows({side: ids for side in sides}, any_of=True)
        wanted = set(coda_typed_ids(ids, False)) | {str(i) for i in coda_typed_ids(ids, True)}
        frames = []
        for side in sides:
            other = 'post' if side == 'pre' else 'pre'
            frames.append(rows[rows[side].isin(wanted)]
                          .rename(columns={side: 'neuronId', other: 'partnerId'})
                          .assign(polarity=side))
        out = pd.concat(frames, ignore_index=True)
        first = ['neuronId', 'partnerId', 'polarity', 'x', 'y', 'z']
        return out[first + [c for c in out.columns if c not in first]]

    def synapses_between(self, sources=None, targets=None, include_fragments=True):
        """Fetch the synapses from sources onto targets; neuronId is the presynaptic end.

        polarity and confidence are left empty: the table does not say which end a position
        belongs to, and has no confidence scale. If only one side is given and
        include_fragments is off, the other end must be in the neuron table (if there is one).
        """
        where = {side: ids for side, ids in (('pre', sources), ('post', targets)) if ids is not None}
        if not where:
            raise ValueError('Synapses between two sets needs sources, targets or both.')
        rows = self._synapse_rows(where, carry=False)
        if not include_fragments and len(where) == 1 and not self._labels_derived:
            open_end = 'post' if 'pre' in where else 'pre'
            rows = rows[rows[open_end].isin(self.labels['neuronId'])].reset_index(drop=True)
        types = self._types()
        return pd.DataFrame({
            'neuronId': rows['pre'].values,
            'type': rows['pre'].map(types).values,
            'partnerId': rows['post'].values,
            'partnerType': rows['post'].map(types).values,
            'polarity': None,
            'confidence': None,
            'x': rows['x'].values,
            'y': rows['y'].values,
            'z': rows['z'].values,
        })

    # ---- geometry --------------------------------------------------------------------------

    def _geometry(self, source, what, ids):
        if source is None:
            raise ValueError(f'This Custom Dataset has no {what} in this notebook: nothing is '
                             f'wired into its {what.capitalize()} socket, or what is wired there '
                             f'has no notebook form (the Custom Dataset cell says so).')
        return source(ids)

    def meshes(self, ids):
        return self._geometry(self.mesh_source, 'meshes', ids)

    def skeletons(self, ids):
        return self._geometry(self.skeleton_source, 'skeletons', ids)


def coda_mesh(*args, **kwargs):
    """Make a navis mesh neuron (`navis.Mesh` in navis 2, `navis.MeshNeuron` before)."""
    return (getattr(navis, 'Mesh', None) or navis.MeshNeuron)(*args, **kwargs)


class CodaPrecomputed:
    """Fetch meshes and skeletons from a Neuroglancer precomputed bucket via cloudvolume.

    Returns navis neurons in nanometres. Segments missing from the bucket are skipped and
    counted, as on the canvas.
    """

    def __init__(self, url):
        self.url = url
        self._cv = None

    @property
    def cv(self):
        if self._cv is None:
            self._cv = CloudVolume(self.url, use_https=True, progress=False)
        return self._cv

    def _each(self, ids, fetch, what):
        # Fetch the whole batch at once; if it fails, split it in half and retry, so a few
        # missing segments cost a few extra requests, not one request per id.
        def attempt(batch):
            try:
                return fetch(batch), []
            except Exception as error:
                if len(batch) == 1:
                    errors.append(error)
                    return {}, batch
                half = len(batch) // 2
                (a, lost_a), (b, lost_b) = attempt(batch[:half]), attempt(batch[half:])
                return {**a, **b}, lost_a + lost_b

        errors = []
        ids = [int(i) for i in ids]
        found, missing = attempt(ids) if ids else ({}, [])
        if missing:
            # Usually segments absent from the bucket; the last error shows if not.
            print(f'{len(missing)} of {len(ids)} segments could not be read as {what} from '
                  f'{self.url} (last error: {errors[-1]!r})')
        return found

    def meshes(self, ids):
        found = self._each(ids, lambda batch: dict(self.cv.mesh.get(batch)), 'mesh')
        return navis.NeuronList(
            [coda_mesh(mesh, id=i, units="nm") for i, mesh in found.items()]
        )

    def skeletons(self, ids):
        def fetch(batch):
            return {int(s.id): s for s in self.cv.skeleton.get(batch)}

        neurons = []
        for i, sk in self._each(ids, fetch, 'skeleton').items():
            tree = navis.edges2neuron(sk.edges, vertices=sk.vertices, validate=True)
            tree.id = i
            tree.units = 'nm'
            neurons.append(tree)
        return navis.NeuronList(neurons)


def coda_endpoint_neurons(connections, seed_ids=None):
    """List the neurons in an edge list: the seeds, then every partner, one row each.

    This is the Neuron Set output of Coda's Connectivity node. Seeds are always included,
    even those left without edges after min_weight.
    """
    frames = []
    if seed_ids is not None:
        frames.append(
            coda_ids(pd.DataFrame({'neuronId': list(seed_ids), 'type': None}), 'neuronId')
        )
    connections = coda_ids(connections.copy(), 'preId', 'postId')
    for id_col, type_col in (('preId', 'preType'), ('postId', 'postType')):
        frames.append(
            connections[[id_col, type_col]].rename(
                columns={id_col: 'neuronId', type_col: 'type'}
            )
        )

    rows = pd.concat(frames, ignore_index=True)
    rows['type'] = rows['type'].replace('', None)
    # Order by first appearance, but take the first non-empty type, which may come from
    # a later row (e.g. an untyped seed typed by an edge).
    typed = (
        rows.dropna(subset=['type'])
        .drop_duplicates(subset='neuronId')
        .set_index('neuronId')['type']
    )
    out = rows.drop_duplicates(subset='neuronId').reset_index(drop=True)
    out['type'] = out['neuronId'].map(typed)
    return out


import re

_CODA_OPERATORS = [("==", "eq"), ("!=", "ne"), (">=", "ge"), ("<=", "le"),
                   ("~", "match"), (">", "gt"), ("<", "lt"), ("=", "eq")]
_CODA_FIELD_NAME = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _coda_tokenize(text):
    """Split on whitespace, keeping quoted text together."""
    tokens, i = [], 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        start, quote = i, None
        while i < len(text):
            ch = text[i]
            if quote is not None:
                if ch == quote:
                    quote = None
            elif ch in "\"'":
                quote = ch
            elif ch.isspace():
                break
            i += 1
        tokens.append(text[start:i])
    return tokens


def _coda_unquote(value):
    if value[:1] in ("\"", "'") and len(value) >= 2:
        return value[1:-1] if value.endswith(value[0]) else value[1:]
    return value


def _coda_split_operator(token):
    """Split a token into (field, operator, value), or return None for a bare word.

    Longer operators are tried first so "!=" is not read as "=". The field must look like
    a column name, so "LC4-a" is not parsed as a comparison.
    """
    for symbol, op in _CODA_OPERATORS:
        at = token.find(symbol)
        if at <= 0:
            continue
        field = token[:at]
        if not _CODA_FIELD_NAME.match(field):
            continue
        return field, op, token[at + len(symbol):]
    return None


def _coda_parse_search(text):
    terms = []
    for raw in _coda_tokenize(text):
        negate = False
        if raw[:1] in ("!", "-") and len(raw) > 1 and _coda_split_operator(raw) is None:
            negate, raw = True, raw[1:]
        split = _coda_split_operator(raw)
        if split is None:
            value = _coda_unquote(raw)
            if value.startswith("/"):
                # A regex, as in neuroglancer. The closing slash is optional; a lone "/"
                # matches everything.
                pattern = value[1:]
                if pattern.endswith("/") and not pattern.endswith("\\/"):
                    pattern = pattern[:-1]
                if pattern:
                    # Not lowercased: the regex is matched case-insensitively instead,
                    # and lowercasing would change patterns like "[A-Z]".
                    terms.append(("regex", pattern, None, None, negate))
                continue
            if value:
                terms.append(("text", value.lower(), None, None, negate))
            continue
        field, op, value = split
        value = _coda_unquote(value)
        if not value:
            # A comparison without a value (e.g. "type=") is ignored.
            continue
        terms.append(("field", value, field, op, negate))
    return terms


def _coda_searchable(df):
    """Return the columns searched by free text and regex terms.

    Only text columns and neuronId, so "1200" finds a neuron id but not every neuron
    with 1200 synapses.
    """
    # Check both: pandas 3 gives text its own `str` dtype, which is not `object`.
    return [c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or str(c) == "neuronId"]


def _coda_haystack(df):
    """Join the searchable columns into one lowercase string per row."""
    cols = _coda_searchable(df)
    if not cols:
        return pd.Series([""] * len(df), index=df.index)
    parts = [df[c].fillna("").astype(str) for c in cols]
    joined = parts[0]
    for part in parts[1:]:
        joined = joined.str.cat(part, sep=" ")
    return joined.str.lower()


def _coda_regex_mask(df, pattern):
    """Test whether any searchable field of a row matches a regex.

    Each field is tested on its own, so anchored patterns like "^LC4$" work. Missing
    values never match.
    """
    rx = re.compile(pattern, re.IGNORECASE)
    mask = pd.Series(False, index=df.index)
    for col in _coda_searchable(df):
        series = df[col]
        present = series.notna()
        text = series.astype(str).where(present, "")
        mask |= text.map(lambda v: rx.search(v) is not None) & present
    return mask


def _coda_field_mask(df, field, op, value):
    """Compare one column against a value.

    A missing value satisfies "!=" and nothing else, so status!=Traced returns both the
    untraced and the unlabelled neurons (unlike SQL, which would drop the unlabelled).
    """
    col = next((c for c in df.columns if str(c).lower() == field.lower()), None)
    if col is None:
        return pd.Series(False, index=df.index)
    series = df[col]
    missing = series.isna()

    if op == "match":
        # Unanchored, unlike neuPrint's "=~".
        rx = re.compile(value)
        found = series.fillna("").astype(str).map(lambda v: rx.search(v) is not None)
        return found & ~missing

    if pd.api.types.is_numeric_dtype(series):
        try:
            right = float(value)
        except ValueError:
            return pd.Series(False, index=df.index)
        left = pd.to_numeric(series, errors="coerce")
    else:
        right = value.lower()
        left = series.fillna("").astype(str).str.lower()

    if op == "eq":
        mask = left == right
    elif op == "ne":
        mask = left != right
    elif op == "gt":
        mask = left > right
    elif op == "lt":
        mask = left < right
    elif op == "ge":
        mask = left >= right
    else:
        mask = left <= right

    mask = mask.fillna(False).astype(bool)
    return (mask | missing) if op == "ne" else (mask & ~missing)


def coda_search(df, query):
    """Filter rows with Coda's Explore Dataset query language.

    Terms are combined with AND; a leading "!" or "-" negates a term. A bare word matches
    as a substring of any text column. A term starting with "/" is a regex tested against
    each text column, e.g. "/^LC4$". "field=value" compares one column; the other
    operators are ">", "<", ">=", "<=", "!=" and "~" (unanchored regex).

    Two differences from Coda, both of which can change which rows you get:

    * Matches come back in table order. Coda ranks them by relevance, which decides
      which rows are kept when the result is capped.
    * A query matching nothing returns nothing. Coda then retries it as a fuzzy match,
      so "mechnosensory" finds "mechanosensory" there but not here. Regex terms are not
      retried in Coda either.
    """
    terms = _coda_parse_search(query)
    if not terms:
        return df

    keep = pd.Series(True, index=df.index)
    haystack = None
    for kind, value, field, op, negate in terms:
        if kind == "text":
            if haystack is None:
                haystack = _coda_haystack(df)
            mask = haystack.str.contains(value, regex=False)
        elif kind == "regex":
            mask = _coda_regex_mask(df, value)
        else:
            mask = _coda_field_mask(df, field, op, value)
        keep &= ~mask if negate else mask

    return df[keep]

In [ ]:
# ── Table from URL ──
table_from_url = pd.read_csv('https://example.org/cell_types.csv')

In [ ]:
# ── Link Table ──
# NOTE: The canvas had not read this table’s log at export, so only integer
# columns whose names say they hold ids are read as text, as Coda does. Add
# any other id column to text_columns, because an eighteen-digit id read as
# a number becomes a different neuron. No version is pinned either, so this
# reads the table as it is when the cell runs.
link_table = CodaTableFile(
    'gs://example-bucket/connectome/edges',
    format='delta',
)

In [ ]:
# ── Link Table ──
# NOTE: This file was on the computer of whoever built this workflow. Set
# the path to your own copy of synapses.parquet.
# NOTE: The canvas had not read this file’s footer at export, so only
# integer columns whose names say they hold ids are read as text, as Coda
# does. Add any other id column to text_columns, because an eighteen-digit
# id read as a number becomes a different neuron.
link_table_2 = CodaTableFile(
    'synapses.parquet',
    format='parquet',
)

In [ ]:
# ── Neuroglancer Source ──
neuroglancer_source_dataset = CodaPrecomputed('precomputed://gs://flywire_v141_m783')
# NOTE: Neuroglancer layers cannot be exported to a notebook, so `Layers` is
# None.
neuroglancer_source_layers = None

In [ ]:
# ── Filter Table ──
filter_table = link_table_2.where(['score'], lambda df: (df['score'] >= 0.5) if pd.api.types.is_numeric_dtype(df['score']) else (df['score'].fillna('').astype(str) == '0.5'))

In [ ]:
# ── Custom Dataset ──
custom_dataset = CodaCustomDataset(
    neurons=table_from_url,
    id_column='neuronId',
    edges={
        'source': link_table,
        'pre': 'pre',
        'post': 'post',
        'weight': 'weight',
    },
    synapses={
        'source': filter_table,
        'pre': 'pre_pt_root_id',
        'post': 'post_pt_root_id',
        'position': ['x', 'y', 'z'],
        'voxel': [1, 1, 1],
        'carry': [],
    },
    meshes=lambda ids: neuroglancer_source_dataset.meshes(ids),
    skeletons=lambda ids: neuroglancer_source_dataset.skeletons(ids),
)

In [ ]:
# ── Find Neurons ──
# NOTE: A Custom Dataset’s neurons are its Neurons table, filtered here as
# on the canvas.
find_neurons = custom_dataset.labels
find_neurons = find_neurons[(find_neurons['type'].notna() & find_neurons['type'].astype(str).str.contains('^(?:LC.*)$', regex=True, case=True))]

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset searches the whole neuron table locally: here the
# Custom Dataset’s Neurons table, or every id its edge list names where none
# is wired.
explore_dataset_all_ = custom_dataset.labels

explore_dataset_hits = coda_search(explore_dataset_all_, 'LC4')

# NOTE: Nothing is ticked on the canvas, so Selected is empty.
explore_dataset_selected = explore_dataset_all_.iloc[0:0]

In [ ]:
# ── Input IDs ──
_ids = [1, 2, 3]
input_ids = custom_dataset.lookup(_ids)

In [ ]:
# ── Connectivity ──
connectivity_connections = custom_dataset.connectivity(
    find_neurons['neuronId'].tolist(),
    direction='outputs',
    hops=1,
    min_weight=1,
    include_fragments=False,
)

_endpoints = coda_endpoint_neurons(connectivity_connections, find_neurons['neuronId'].tolist())
connectivity_neuron_set = _endpoints[['neuronId']].merge(custom_dataset.labels, on='neuronId')

In [ ]:
# ── Connectivity ──
connectivity_2_connections = custom_dataset.connectivity(
    find_neurons['neuronId'].tolist(),
    direction='both',
    hops=2,
    min_weight=1,
    include_fragments=True,
)

connectivity_2_neuron_set = coda_endpoint_neurons(connectivity_2_connections, find_neurons['neuronId'].tolist())

In [ ]:
# ── Synapses ──
synapses = custom_dataset.synapses(
    find_neurons['neuronId'].tolist(),
    polarity='pre',
)

In [ ]:
# ── Skeletons ──
skeletons = custom_dataset.skeletons(find_neurons['neuronId'].head(10000).astype('int64').tolist())

In [ ]:
# ── Meshes ──
meshes = neuroglancer_source_dataset.meshes(find_neurons['neuronId'].head(10000).astype('int64').tolist())

In [ ]:
# ── Read Rows ──
_ids = []
_ids += find_neurons['neuronId'].dropna().tolist()
read_rows = link_table_2.read(
    columns=['pre_pt_root_id', 'x', 'y', 'z'],
    where={'pre_pt_root_id': _ids},
    limit=1000001,
)
if len(read_rows) > 1000000:
    print('Stopped at the row cap of 1,000,000 with more rows left to read. '
          'Raise the cap, match fewer ids, or filter the table upstream.')
    read_rows = read_rows.head(1000000)

In [ ]:
# ── Synapses Between ──
# NOTE: A synapse table has one position per synapse and does not say which
# end it is at, so `Location` has no effect and polarity is left empty, as
# on the canvas.
synapses_between = custom_dataset.synapses_between(
    sources=find_neurons['neuronId'].tolist(),
    targets=input_ids['neuronId'].tolist(),
)